In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)

In [11]:
df = pd.read_csv(
    "../data/processed/final_zomato.csv"
)

df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city),votes_norm,online_order_num,book_table_num,success_score
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800.0,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari,0.047415,1,1,2.264225
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800.0,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari,0.048149,1,0,2.164445
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800.0,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari,0.056164,1,0,2.016849
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300.0,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari,0.005384,0,0,1.851615
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600.0,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari,0.010156,0,0,1.903047


In [22]:
model_df = df[
    [
        "rate",
        "votes",
        "approx_cost(for two people)",
        "online_order_num",
        "book_table_num",
        "location",
        "rest_type",
        "listed_in(type)",
        "listed_in(city)",
        "cuisines"
    ]
].copy()

model_df.head()

,rate,votes,approx_cost(for two people),online_order_num,book_table_num,location,rest_type,listed_in(type),listed_in(city),cuisines
0,4.1,775,800.0,1,1,Banashankari,Casual Dining,Buffet,Banashankari,"North Indian, Mughlai, Chinese"
1,4.1,787,800.0,1,0,Banashankari,Casual Dining,Buffet,Banashankari,"Chinese, North Indian, Thai"
2,3.8,918,800.0,1,0,Banashankari,"Cafe, Casual Dining",Buffet,Banashankari,"Cafe, Mexican, Italian"
3,3.7,88,300.0,0,0,Banashankari,Quick Bites,Buffet,Banashankari,"South Indian, North Indian"
4,3.8,166,600.0,0,0,Basavanagudi,Casual Dining,Buffet,Banashankari,"North Indian, Rajasthani"


In [23]:
model_df.dropna(inplace=True)

model_df.shape

(9284, 10)

In [24]:
model_df["cuisine_count"] = (
    model_df["cuisines"]
    .apply(
        lambda x: len(str(x).split(","))
    )
)

In [25]:
rest_popularity = (
    model_df
    .groupby("rest_type")["votes"]
    .mean()
)

model_df["rest_popularity"] = (
    model_df["rest_type"]
    .map(rest_popularity)
)

In [26]:
from sklearn.preprocessing import LabelEncoder

le_location = LabelEncoder()
le_rest = LabelEncoder()
le_type = LabelEncoder()
le_city = LabelEncoder()

model_df["location_encoded"] = (
    le_location.fit_transform(
        model_df["location"]
    )
)

model_df["rest_type_encoded"] = (
    le_rest.fit_transform(
        model_df["rest_type"]
    )
)

model_df["listed_type_encoded"] = (
    le_type.fit_transform(
        model_df["listed_in(type)"]
    )
)

model_df["listed_city_encoded"] = (
    le_city.fit_transform(
        model_df["listed_in(city)"]
    )
)

In [27]:
features = [
    "votes",
    "approx_cost(for two people)",
    "online_order_num",
    "book_table_num",
    "location_encoded",
    "rest_type_encoded",
    "listed_type_encoded",
    "listed_city_encoded",
    "cuisine_count",
    "rest_popularity"
]

X = model_df[features]

y = model_df["rate"]

print(X.shape)
print(y.shape)

(9284, 10)
(9284,)


In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [30]:
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [31]:
y_pred = rf.predict(
    X_test
)

In [32]:
print(
    "MAE:",
    round(
        mean_absolute_error(
            y_test,
            y_pred
        ),
        4
    )
)

print(
    "RMSE:",
    round(
        np.sqrt(
            mean_squared_error(
                y_test,
                y_pred
            )
        ),
        4
    )
)

print(
    "R2 Score:",
    round(
        r2_score(
            y_test,
            y_pred
        ),
        4
    )
)

MAE: 0.2412
RMSE: 0.333
R2 Score: 0.4162


In [33]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = (
    importance
    .sort_values(
        "Importance",
        ascending=False
    )
)

importance

,Feature,Importance
0,votes,0.541563
1,approx_cost(for two people),0.106169
4,location_encoded,0.097020
7,listed_city_encoded,0.070710
5,rest_type_encoded,0.051549
8,cuisine_count,0.045510
9,rest_popularity,0.038582
6,listed_type_encoded,0.021483
2,online_order_num,0.016847
3,book_table_num,0.010567


In [34]:
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

comparison.head(20)

,Actual,Predicted
165,3.2,3.898756
7987,4.0,4.019391
886,4.1,3.861421
3755,3.8,3.513356
1232,3.3,3.285481
3861,4.0,3.799419
10419,3.7,3.807173
2757,3.2,3.222755
4795,4.4,4.264504
6935,3.3,3.203394


In [36]:
import joblib

joblib.dump(
    rf,
    "../models/rating_model.pkl"
)

print(
    "Model Saved Successfully"
)

Model Saved Successfully
